In [3]:
import pandas as pd
import numpy as np


In [4]:
DATA_PATH = "../data/processed/paysim_features.csv"

df = pd.read_csv(DATA_PATH)

print("shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

shape: (6362620, 24)

Columns:
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'log_amount', 'hour', 'day', 'origin_balance_utilization', 'origin_balance_zero', 'destination_balance_zero', 'full_origin_balance_transfer', 'destination_previous_transactions', 'origin_previous_transactions', 'origin_balance_depleted', 'destination_balance_change', 'origin_balance_change', 'amount_to_destination_balance', 'zero_amount_transaction']


In [5]:
# ============================================================
# 2. LOAD SAVED MODEL AND PREPROCESSOR
# ============================================================

import joblib

MODEL_PATH = "../models/xgboost_fraud_model.pkl"
PREPROCESSOR_PATH = "../models/preprocessor.pkl"

# Load trained XGBoost model
final_model = joblib.load(MODEL_PATH)

# Load preprocessing pipeline
preprocessor = joblib.load(PREPROCESSOR_PATH)

print("Model loaded successfully.")
print("Model type:", type(final_model).__name__)

print("\nPreprocessor loaded successfully.")
print("Preprocessor type:", type(preprocessor).__name__)

Model loaded successfully.
Model type: XGBClassifier

Preprocessor loaded successfully.
Preprocessor type: ColumnTransformer


In [7]:
# ============================================================
# 3. PREPARE MODEL INPUT FEATURES
# ============================================================

# Columns that must NOT enter the model
# - isFraud   → target variable
# - nameOrig  → transaction identifier
# - nameDest  → transaction identifier

DROP_COLUMNS = ["isFraud","nameOrig","nameDest"]

X_risk = df.drop(columns=DROP_COLUMNS)

print("Model input columns:")
print(X_risk.columns.tolist())



Model input columns:
['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'log_amount', 'hour', 'day', 'origin_balance_utilization', 'origin_balance_zero', 'destination_balance_zero', 'full_origin_balance_transfer', 'destination_previous_transactions', 'origin_previous_transactions', 'origin_balance_depleted', 'destination_balance_change', 'origin_balance_change', 'amount_to_destination_balance', 'zero_amount_transaction']


In [11]:
# ============================================================
# 4. TRANSFORM FEATURES USING SAVED PREPROCESSOR
# ============================================================

X_risk_processed = preprocessor.transform(X_risk)

print("Original feature shape:", X_risk.shape)
print("Processed feature shape:", X_risk_processed.shape)

print("\nProcessed feature type:", type(X_risk_processed).__name__)

Original feature shape: (6362620, 21)
Processed feature shape: (6362620, 25)

Processed feature type: ndarray


In [13]:
# ============================================================
# 5. GENERATE FRAUD PROBABILITIES
# ============================================================

CHUNK_SIZE = 500_000

risk_probabilities = []

total_rows = X_risk_processed.shape[0]

for start in range(0, total_rows, CHUNK_SIZE):
    
    end = min(start + CHUNK_SIZE, total_rows)
    
    X_chunk = X_risk_processed[start:end]
    
    # Probability of class 1 = fraud probability
    chunk_probability = final_model.predict_proba(X_chunk)[:, 1]
    
    risk_probabilities.append(chunk_probability)
    
    print(f"Processed rows: {end:,} / {total_rows:,}")

# Combine all chunks
risk_probabilities = np.concatenate(risk_probabilities)

print("\nPrediction completed.")
print("Number of probabilities:", len(risk_probabilities))
print("Minimum probability:", risk_probabilities.min())
print("Maximum probability:", risk_probabilities.max())
print("Mean probability:", risk_probabilities.mean())

Processed rows: 500,000 / 6,362,620
Processed rows: 1,000,000 / 6,362,620
Processed rows: 1,500,000 / 6,362,620
Processed rows: 2,000,000 / 6,362,620
Processed rows: 2,500,000 / 6,362,620
Processed rows: 3,000,000 / 6,362,620
Processed rows: 3,500,000 / 6,362,620
Processed rows: 4,000,000 / 6,362,620
Processed rows: 4,500,000 / 6,362,620
Processed rows: 5,000,000 / 6,362,620
Processed rows: 5,500,000 / 6,362,620
Processed rows: 6,000,000 / 6,362,620
Processed rows: 6,362,620 / 6,362,620

Prediction completed.
Number of probabilities: 6362620
Minimum probability: 0.007312837
Maximum probability: 0.9940639
Mean probability: 0.010975927


In [16]:
# ============================================================
# 6. ATTACH RISK PROBABILITY & INSPECT DISTRIBUTION
# ============================================================

# Add model-generated fraud probability to the original dataset
df["fraud_probability"] = risk_probabilities

# basic probability statistics
print("Fraud Probability Statistics:")
print(df["fraud_probability"].describe())

print("\nSelected Probability Percentiles:")
print(
    df["fraud_probability"].quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999, 0.9995, 0.9999]
    )
)

Fraud Probability Statistics:
count    6.362620e+06
mean     1.097593e-02
std      3.599484e-02
min      7.312837e-03
25%      8.064734e-03
50%      8.334576e-03
75%      9.568365e-03
max      9.940639e-01
Name: fraud_probability, dtype: float64

Selected Probability Percentiles:
0.5000    0.008335
0.7500    0.009568
0.9000    0.011318
0.9500    0.015220
0.9900    0.028975
0.9950    0.043491
0.9990    0.991868
0.9995    0.992001
0.9999    0.992002
Name: fraud_probability, dtype: float64


In [17]:
# ============================================================
# 7. ANALYZE RISK PROBABILITY SEGMENTS
# ============================================================

# Define probability ranges for distribution analysis
risk_bins = [0, 0.01, 0.02, 0.05, 0.10, 0.50, 0.90, 1.00]

risk_labels = ["0-1%", "1-2%", "2-5%", "5-10%", "10-50%", "50-90%", "90-100%"]

risk_distribution = pd.cut(
    df["fraud_probability"],
    bins=risk_bins,
    labels=risk_labels,
    include_lowest=True
)

risk_summary = (
    risk_distribution
    .value_counts(sort=False)
    .rename_axis("Probability Range")
    .reset_index(name="Transactions")
)

risk_summary["Percentage"] = (
    risk_summary["Transactions"]
    / len(df)
    * 100
)

print(risk_summary)

  Probability Range  Transactions  Percentage
0              0-1%       4996706   78.532208
1              1-2%       1194438   18.772738
2              2-5%        145107    2.280617
3             5-10%         14498    0.227862
4            10-50%          3496    0.054946
5            50-90%           193    0.003033
6           90-100%          8182    0.128595


In [18]:
# ============================================================
# 7. CREATE DATA-DRIVEN RISK BANDS
# ============================================================

RISK_BINS = [ 0, 0.01, 0.02, 0.10, 1.00]

RISK_LABELS = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]

df["risk_band"] = pd.cut(
    df["fraud_probability"],
    bins=RISK_BINS,
    labels=RISK_LABELS,
    include_lowest=True,
    right=False
)

# Summarize risk bands
risk_band_summary = (
    df["risk_band"]
    .value_counts(sort=False)
    .rename_axis("Risk Band")
    .reset_index(name="Transactions")
)

risk_band_summary["Percentage"] = (
    risk_band_summary["Transactions"]
    / len(df)
    * 100
)

print(risk_band_summary)

  Risk Band  Transactions  Percentage
0       LOW       4996706   78.532208
1    MEDIUM       1194438   18.772738
2      HIGH        159605    2.508479
3  CRITICAL         11871    0.186574


In [19]:
# ============================================================
# 8. CREATE TRANSACTION RISK SCORE
# ============================================================

# Convert model probability into a 0–100 risk score
df["risk_score"] = df["fraud_probability"] * 100

# Inspect risk score statistics
print("Risk Score Statistics:")
print(df["risk_score"].describe())

print("\nSample Risk Scores:")
print(
    df[
        [
            "fraud_probability",
            "risk_score",
            "risk_band"
        ]
    ].head(10)
)

Risk Score Statistics:
count    6.362620e+06
mean     1.097593e+00
std      3.599484e+00
min      7.312837e-01
25%      8.064734e-01
50%      8.334576e-01
75%      9.568365e-01
max      9.940639e+01
Name: risk_score, dtype: float64

Sample Risk Scores:
   fraud_probability  risk_score risk_band
0           0.009906    0.990601       LOW
1           0.009906    0.990601       LOW
2           0.991923   99.192276  CRITICAL
3           0.991191   99.119080  CRITICAL
4           0.009906    0.990601       LOW
5           0.009906    0.990601       LOW
6           0.009906    0.990601       LOW
7           0.009906    0.990601       LOW
8           0.013685    1.368528    MEDIUM
9           0.009906    0.990601       LOW


In [20]:
# ============================================================
# 9. CREATE INVESTIGATION PRIORITY
# ============================================================

INVESTIGATION_BINS = [ 0, 0.90,  0.95, 1.00]

INVESTIGATION_LABELS = [ "ROUTINE", "REVIEW", "URGENT"]

df["investigation_priority"] = pd.cut(
    df["fraud_probability"],
    bins=INVESTIGATION_BINS,
    labels=INVESTIGATION_LABELS,
    include_lowest=True,
    right=False
)

# Summarize investigation priorities
priority_summary = (
    df["investigation_priority"]
    .value_counts(sort=False)
    .rename_axis("Investigation Priority")
    .reset_index(name="Transactions")
)

priority_summary["Percentage"] = (
    priority_summary["Transactions"]
    / len(df)
    * 100
)

print(priority_summary)

  Investigation Priority  Transactions  Percentage
0                ROUTINE       6354438   99.871405
1                 REVIEW             1    0.000016
2                 URGENT          8181    0.128579


In [21]:
# ============================================================
# 10. CREATE EXPLAINABLE RISK SIGNALS
# ============================================================

# Create individual rule-based risk signals

df["signal_full_balance_transfer"] = ( df["full_origin_balance_transfer"] == 1 )

df["signal_balance_depleted"] = ( df["origin_balance_depleted"] == 1 )

df["signal_zero_amount"] = ( df["zero_amount_transaction"] == 1 )

df["signal_high_balance_utilization"] = ( df["origin_balance_utilization"] >= 0.90 )

df["signal_large_origin_balance_change"] = ( df["origin_balance_change"].abs() >= df["amount"] )

# Count the number of triggered risk signals
signal_columns = [
    "signal_full_balance_transfer",
    "signal_balance_depleted",
    "signal_zero_amount",
    "signal_high_balance_utilization",
    "signal_large_origin_balance_change"
]

df["risk_signal_count"] = df[signal_columns].sum(axis=1)

print("Risk signal distribution:")
print(
    df["risk_signal_count"]
    .value_counts()
    .sort_index()
)

print("\nMaximum risk signals triggered:",
      df["risk_signal_count"].max())

Risk signal distribution:
risk_signal_count
0    2890954
1    1446467
2    2017191
4       8008
Name: count, dtype: int64

Maximum risk signals triggered: 4


compare the number of triggered signals with:

Average model probability, 
Number of critical transactions, 
Percentage of transactions classified as critical


In [22]:
# ============================================================
# 11. VALIDATE RISK SIGNALS AGAINST MODEL RISK
# ============================================================

signal_risk_summary = (
    df.groupby("risk_signal_count", observed=True)
    .agg(
        transactions=("fraud_probability", "size"),
        average_probability=("fraud_probability", "mean"),
        maximum_probability=("fraud_probability", "max"),
        critical_transactions=(
            "fraud_probability",
            lambda x: (x >= 0.10).sum()
        )
    )
    .reset_index()
)

signal_risk_summary["critical_percentage"] = (
    signal_risk_summary["critical_transactions"]
    / signal_risk_summary["transactions"]
    * 100
)

print(signal_risk_summary)

   risk_signal_count  transactions  average_probability  maximum_probability  \
0                  0       2890954             0.010507             0.992290   
1                  1       1446467             0.008502             0.992664   
2                  2       2017191             0.009528             0.994064   
3                  4          8008             0.991903             0.992002   

   critical_transactions  critical_percentage  
0                   2570             0.088898  
1                    180             0.012444  
2                   1113             0.055176  
3                   8008           100.000000  


In [23]:
# ============================================================
# 12. IDENTIFY RISK-SIGNAL COMBINATIONS
# ============================================================

signal_combination_summary = (
    df.groupby(
        signal_columns,
        observed=True
    )
    .agg(
        transactions=("fraud_probability", "size"),
        average_probability=("fraud_probability", "mean"),
        maximum_probability=("fraud_probability", "max"),
        critical_transactions=(
            "fraud_probability",
            lambda x: (x >= 0.10).sum()
        )
    )
    .reset_index()
)

signal_combination_summary["critical_percentage"] = (
    signal_combination_summary["critical_transactions"]
    / signal_combination_summary["transactions"]
    * 100
)

# Sort by model risk
signal_combination_summary = (
    signal_combination_summary
    .sort_values(
        "average_probability",
        ascending=False
    )
)

print(signal_combination_summary.to_string(index=False))

 signal_full_balance_transfer  signal_balance_depleted  signal_zero_amount  signal_high_balance_utilization  signal_large_origin_balance_change  transactions  average_probability  maximum_probability  critical_transactions  critical_percentage
                         True                     True               False                             True                                True          8008             0.991903             0.992002                   8008           100.000000
                        False                    False                True                            False                                True            16             0.989139             0.991138                     16           100.000000
                         True                    False               False                             True                               False            10             0.986583             0.994064                     10           100.000000
                        

In [24]:
# ============================================================
# 13. CREATE RISK INTELLIGENCE DATASET
# ============================================================

# Select core transaction information
risk_intelligence = df[
    [
        "step",
        "type",
        "amount",
        "nameOrig",
        "nameDest",
        "isFraud",
        "fraud_probability",
        "risk_score",
        "risk_band",
        "investigation_priority",
        "risk_signal_count",
        "signal_full_balance_transfer",
        "signal_balance_depleted",
        "signal_zero_amount",
        "signal_high_balance_utilization",
        "signal_large_origin_balance_change"
    ]
].copy()

print("Risk Intelligence Shape:", risk_intelligence.shape)

print("\nColumns:")
print(risk_intelligence.columns.tolist())

print("\nSample:")
display(risk_intelligence.head())

Risk Intelligence Shape: (6362620, 16)

Columns:
['step', 'type', 'amount', 'nameOrig', 'nameDest', 'isFraud', 'fraud_probability', 'risk_score', 'risk_band', 'investigation_priority', 'risk_signal_count', 'signal_full_balance_transfer', 'signal_balance_depleted', 'signal_zero_amount', 'signal_high_balance_utilization', 'signal_large_origin_balance_change']

Sample:


,step,type,amount,nameOrig,nameDest,isFraud,fraud_probability,risk_score,risk_band,investigation_priority,risk_signal_count,signal_full_balance_transfer,signal_balance_depleted,signal_zero_amount,signal_high_balance_utilization,signal_large_origin_balance_change
0,1,PAYMENT,9839.64,C1231006815,M1979787155,0,0.009906,0.990601,LOW,ROUTINE,1,False,False,False,False,True
1,1,PAYMENT,1864.28,C1666544295,M2044282225,0,0.009906,0.990601,LOW,ROUTINE,0,False,False,False,False,False
2,1,TRANSFER,181.00,C1305486145,C553264065,1,0.991923,99.192276,CRITICAL,URGENT,4,True,True,False,True,True
3,1,CASH_OUT,181.00,C840083671,C38997010,1,0.991191,99.119080,CRITICAL,URGENT,4,True,True,False,True,True
4,1,PAYMENT,11668.14,C2048537720,M1230701703,0,0.009906,0.990601,LOW,ROUTINE,1,False,False,False,False,True


In [25]:
# ============================================================
# 14. VALIDATE RISK BANDS AGAINST ACTUAL FRAUD
# ============================================================

risk_band_validation = (
    risk_intelligence.groupby("risk_band", observed=True)
    .agg(
        transactions=("isFraud", "size"),
        fraud_transactions=("isFraud", "sum"),
        total_amount=("amount", "sum"),
        average_amount=("amount", "mean")
    )
    .reset_index()
)

# Calculate actual fraud rate within each risk band
risk_band_validation["fraud_rate"] = (
    risk_band_validation["fraud_transactions"]
    / risk_band_validation["transactions"]
    * 100
)

# Calculate how much of all fraud each band captures
total_fraud = risk_intelligence["isFraud"].sum()

risk_band_validation["fraud_capture"] = (
    risk_band_validation["fraud_transactions"]
    / total_fraud
    * 100
)

# Calculate actual fraudulent transaction amount
fraud_amounts = (
    risk_intelligence[
        risk_intelligence["isFraud"] == 1
    ]
    .groupby("risk_band", observed=True)["amount"]
    .sum()
)

risk_band_validation["fraud_amount"] = (
    risk_band_validation["risk_band"]
    .map(fraud_amounts)
    .fillna(0)
)

print("Risk Band Validation:")
display(risk_band_validation)

Risk Band Validation:


,risk_band,transactions,fraud_transactions,total_amount,average_amount,fraud_rate,fraud_capture,fraud_amount
0,LOW,4996706,1,7.931071e+11,1.587260e+05,0.000020,0.012176,1.231949e+05
1,MEDIUM,1194438,4,2.827413e+11,2.367149e+05,0.000335,0.048703,5.628809e+05
2,HIGH,159605,0,5.515648e+10,3.455812e+05,0.000000,0.000000,0.000000e+00
3,CRITICAL,11871,8208,1.338809e+10,1.127798e+06,69.143290,99.939121,1.205573e+10


In [26]:
# ============================================================
# 15. ANALYZE FRAUD CONCENTRATION BY MODEL PROBABILITY
# ============================================================

probability_validation = (
    risk_intelligence
    .assign(
        probability_range=pd.cut(
            risk_intelligence["fraud_probability"],
            bins=[0, 0.01, 0.02, 0.05, 0.10, 0.50, 0.90, 1.00],
            labels=["0-1%", "1-2%", "2-5%", "5-10%", "10-50%", "50-90%", "90-100%"],
            include_lowest=True
        )
    )
    .groupby("probability_range", observed=True)
    .agg(
        transactions=("isFraud", "size"),
        fraud_transactions=("isFraud", "sum"),
        total_amount=("amount", "sum")
    )
    .reset_index()
)

probability_validation["fraud_rate"] = (
    probability_validation["fraud_transactions"]
    / probability_validation["transactions"]
    * 100
)

probability_validation["fraud_capture"] = (
    probability_validation["fraud_transactions"]
    / risk_intelligence["isFraud"].sum()
    * 100
)

print("Fraud Concentration by Model Probability:")
display(probability_validation)

Fraud Concentration by Model Probability:


,probability_range,transactions,fraud_transactions,total_amount,fraud_rate,fraud_capture
0,0-1%,4996706,1,7.931071e+11,0.000020,0.012176
1,1-2%,1194438,4,2.827413e+11,0.000335,0.048703
2,2-5%,145107,0,5.016384e+10,0.000000,0.000000
3,5-10%,14498,0,4.992643e+09,0.000000,0.000000
4,10-50%,3496,13,1.206214e+09,0.371854,0.158286
5,50-90%,193,13,1.366795e+08,6.735751,0.158286
6,90-100%,8182,8182,1.204520e+10,100.000000,99.622550


In [27]:
# ============================================================
# 16. FINALIZE OPERATIONAL RISK TIERS
# ============================================================

OPERATIONAL_BINS = [0, 0.01, 0.10, 0.90, 1.00]

OPERATIONAL_LABELS = [ "LOW", "MEDIUM", "HIGH", "CRITICAL"]

risk_intelligence["operational_risk_tier"] = pd.cut(
    risk_intelligence["fraud_probability"],
    bins=OPERATIONAL_BINS,
    labels=OPERATIONAL_LABELS,
    include_lowest=True,
    right=False
)

operational_summary = (
    risk_intelligence
    .groupby("operational_risk_tier", observed=True)
    .agg(
        transactions=("isFraud", "size"),
        fraud_transactions=("isFraud", "sum"),
        total_amount=("amount", "sum")
    )
    .reset_index()
)

operational_summary["fraud_rate"] = (
    operational_summary["fraud_transactions"]
    / operational_summary["transactions"]
    * 100
)

operational_summary["fraud_capture"] = (
    operational_summary["fraud_transactions"]
    / risk_intelligence["isFraud"].sum()
    * 100
)

print("Final Operational Risk Tier Summary:")
display(operational_summary)

Final Operational Risk Tier Summary:


,operational_risk_tier,transactions,fraud_transactions,total_amount,fraud_rate,fraud_capture
0,LOW,4996706,1,7.931071e+11,0.000020,0.012176
1,MEDIUM,1354043,4,3.378978e+11,0.000295,0.048703
2,HIGH,3689,26,1.342893e+09,0.704798,0.316571
3,CRITICAL,8182,8182,1.204520e+10,100.000000,99.622550


In [28]:
# ============================================================
# 17. CREATE TRANSACTION INVESTIGATION QUEUE
# ============================================================

investigation_queue = (
    risk_intelligence[
        risk_intelligence["operational_risk_tier"].isin(
            ["HIGH", "CRITICAL"]
        )
    ]
    .copy()
)

# Sort highest-risk transactions first
investigation_queue = investigation_queue.sort_values(
    by=["fraud_probability", "amount"],
    ascending=[False, False]
)

# Assign investigation priority rank
investigation_queue["investigation_rank"] = range(
    1,
    len(investigation_queue) + 1
)

print("Investigation Queue Shape:", investigation_queue.shape)

print("\nRisk Tier Distribution:")
print(
    investigation_queue["operational_risk_tier"]
    .value_counts()
    .sort_index()
)

print("\nTop 20 Highest-Risk Transactions:")
display(
    investigation_queue[
        [
            "investigation_rank",
            "step",
            "type",
            "amount",
            "nameOrig",
            "nameDest",
            "fraud_probability",
            "risk_score",
            "operational_risk_tier",
            "risk_signal_count",
            "signal_full_balance_transfer",
            "signal_balance_depleted",
            "signal_zero_amount",
            "signal_high_balance_utilization",
            "signal_large_origin_balance_change",
            "isFraud"
        ]
    ].head(20)
)

Investigation Queue Shape: (11871, 18)

Risk Tier Distribution:
operational_risk_tier
LOW            0
MEDIUM         0
HIGH        3689
CRITICAL    8182
Name: count, dtype: int64

Top 20 Highest-Risk Transactions:


,investigation_rank,step,type,amount,nameOrig,nameDest,fraud_probability,risk_score,operational_risk_tier,risk_signal_count,signal_full_balance_transfer,signal_balance_depleted,signal_zero_amount,signal_high_balance_utilization,signal_large_origin_balance_change,isFraud
6362584,1,741,TRANSFER,5674547.89,C992223106,C1366804249,0.994064,99.406395,CRITICAL,2,True,False,False,True,False,1
2736446,2,212,TRANSFER,4953893.08,C728984460,C639921569,0.994064,99.406395,CRITICAL,2,True,False,False,True,False,1
5563713,3,387,TRANSFER,4892193.09,C908544136,C891140444,0.994064,99.406395,CRITICAL,2,True,False,False,True,False,1
6168499,4,554,TRANSFER,3576297.10,C193696150,C484597480,0.994064,99.406395,CRITICAL,2,True,False,False,True,False,1
6296014,5,671,TRANSFER,3441041.46,C917414431,C1082139865,0.994064,99.406395,CRITICAL,2,True,False,False,True,False,1
6351225,6,702,TRANSFER,3171085.59,C1892216157,C1308068787,0.994064,99.406395,CRITICAL,2,True,False,False,True,False,1
4440,7,4,TRANSFER,10000000.00,C7162498,C945327594,0.992664,99.266373,CRITICAL,1,False,False,False,False,True,1
481250,8,19,TRANSFER,10000000.00,C416779475,C380259496,0.992290,99.228996,CRITICAL,1,False,False,False,False,True,1
586311,9,33,TRANSFER,10000000.00,C1439740840,C875288652,0.992290,99.228996,CRITICAL,0,False,False,False,False,False,1
1030559,10,72,TRANSFER,10000000.00,C53057884,C588547519,0.992091,99.209091,CRITICAL,1,False,False,False,False,True,1


In [29]:
# ============================================================
# 18. INVESTIGATION QUEUE PERFORMANCE
# ============================================================

queue_performance = (
    investigation_queue
    .groupby("operational_risk_tier", observed=True)
    .agg(
        transactions=("isFraud", "size"),
        fraud_transactions=("isFraud", "sum"),
        transaction_amount=("amount", "sum"),
        fraud_amount=("amount", lambda x: x[
            investigation_queue.loc[x.index, "isFraud"] == 1
        ].sum()),
        average_amount=("amount", "mean")
    )
    .reset_index()
)

# Actual fraud rate within each investigation tier
queue_performance["fraud_rate"] = (
    queue_performance["fraud_transactions"]
    / queue_performance["transactions"]
    * 100
)

# Total fraud captured by the investigation queue
total_fraud = risk_intelligence["isFraud"].sum()

queue_performance["fraud_capture"] = (
    queue_performance["fraud_transactions"]
    / total_fraud
    * 100
)

print("Investigation Queue Performance:")
display(queue_performance)

print("\nOverall Investigation Queue:")
print("Transactions:", len(investigation_queue))
print(
    "Percentage of all transactions:",
    len(investigation_queue) / len(risk_intelligence) * 100
)
print(
    "Fraud transactions captured:",
    investigation_queue["isFraud"].sum()
)
print(
    "Fraud capture:",
    investigation_queue["isFraud"].sum()
    / total_fraud * 100
)
print(
    "Transaction amount under investigation:",
    investigation_queue["amount"].sum()
)

Investigation Queue Performance:


,operational_risk_tier,transactions,fraud_transactions,transaction_amount,fraud_amount,average_amount,fraud_rate,fraud_capture
0,HIGH,3689,26,1.342893e+09,1.052842e+07,3.640264e+05,0.704798,0.316571
1,CRITICAL,8182,8182,1.204520e+10,1.204520e+10,1.472159e+06,100.000000,99.622550



Overall Investigation Queue:
Transactions: 11871
Percentage of all transactions: 0.18657408426088623
Fraud transactions captured: 8208
Fraud capture: 99.93912090588091
Transaction amount under investigation: 13388094202.44


In [30]:
# ============================================================
# 19. INVESTIGATION QUEUE — TRANSACTION TYPE ANALYSIS
# ============================================================

queue_type_analysis = (
    investigation_queue
    .groupby(
        ["operational_risk_tier", "type"],
        observed=True
    )
    .agg(
        transactions=("isFraud", "size"),
        fraud_transactions=("isFraud", "sum"),
        transaction_amount=("amount", "sum"),
        fraud_amount=("amount", lambda x: x[
            investigation_queue.loc[x.index, "isFraud"] == 1
        ].sum()),
        average_amount=("amount", "mean")
    )
    .reset_index()
)

queue_type_analysis["fraud_rate"] = (
    queue_type_analysis["fraud_transactions"]
    / queue_type_analysis["transactions"]
    * 100
)

queue_type_analysis["fraud_capture"] = (
    queue_type_analysis["fraud_transactions"]
    / risk_intelligence["isFraud"].sum()
    * 100
)

queue_type_analysis = queue_type_analysis.sort_values(
    by=["operational_risk_tier", "transactions"],
    ascending=[True, False]
)

print("Investigation Queue by Transaction Type:")
display(queue_type_analysis)

Investigation Queue by Transaction Type:


,operational_risk_tier,type,transactions,fraud_transactions,transaction_amount,fraud_amount,average_amount,fraud_rate,fraud_capture
1,HIGH,CASH_OUT,2912,20,6.749707e+08,4.524153e+06,2.317894e+05,0.686813,0.243516
3,HIGH,TRANSFER,735,6,6.585294e+08,6.004262e+06,8.959583e+05,0.816327,0.073055
0,HIGH,CASH_IN,40,0,9.381875e+06,0.000000e+00,2.345469e+05,0.000000,0.000000
2,HIGH,DEBIT,2,0,1.136318e+04,0.000000e+00,5.681590e+03,0.000000,0.000000
4,CRITICAL,CASH_OUT,4092,4092,5.984115e+09,5.984115e+09,1.462394e+06,100.000000,49.823451
5,CRITICAL,TRANSFER,4090,4090,6.061086e+09,6.061086e+09,1.481928e+06,100.000000,49.799099


In [31]:
# ============================================================
# 20. EXECUTIVE RISK INTELLIGENCE KPIs
# ============================================================

total_transactions = len(risk_intelligence)

total_amount = risk_intelligence["amount"].sum()

total_fraud_transactions = risk_intelligence["isFraud"].sum()

total_fraud_amount = risk_intelligence.loc[
    risk_intelligence["isFraud"] == 1,
    "amount"
].sum()

investigation_transactions = len(investigation_queue)

investigation_amount = investigation_queue["amount"].sum()

investigation_fraud_transactions = investigation_queue["isFraud"].sum()

critical_transactions = (
    risk_intelligence["operational_risk_tier"] == "CRITICAL"
).sum()

critical_fraud_transactions = risk_intelligence.loc[
    risk_intelligence["operational_risk_tier"] == "CRITICAL",
    "isFraud"
].sum()

critical_amount = risk_intelligence.loc[
    risk_intelligence["operational_risk_tier"] == "CRITICAL",
    "amount"
].sum()

critical_fraud_amount = risk_intelligence.loc[
    (risk_intelligence["operational_risk_tier"] == "CRITICAL") &
    (risk_intelligence["isFraud"] == 1),
    "amount"
].sum()


executive_kpis = pd.DataFrame({
    "KPI": [
        "Total Transactions",
        "Total Transaction Value",
        "Total Fraud Transactions",
        "Total Fraud Amount",
        "Investigation Transactions",
        "Investigation Queue %",

        "Fraud Capture %",
        "Critical Transactions",
        "Critical Fraud Rate %",
        "Critical Transaction Value",
        "Critical Fraud Value"
    ],

    "Value": [
        total_transactions,
        total_amount,
        total_fraud_transactions,
        total_fraud_amount,
        investigation_transactions,
        investigation_transactions / total_transactions * 100,

        investigation_fraud_transactions /
        total_fraud_transactions * 100,

        critical_transactions,

        critical_fraud_transactions /
        critical_transactions * 100,

        critical_amount,
        critical_fraud_amount
    ]
})

print("Executive Risk Intelligence KPIs:")
display(executive_kpis)

Executive Risk Intelligence KPIs:


,KPI,Value
0,Total Transactions,6.362620e+06
1,Total Transaction Value,1.144393e+12
2,Total Fraud Transactions,8.213000e+03
3,Total Fraud Amount,1.205642e+10
4,Investigation Transactions,1.187100e+04
5,Investigation Queue %,1.865741e-01
6,Fraud Capture %,9.993912e+01
7,Critical Transactions,8.182000e+03
8,Critical Fraud Rate %,1.000000e+02
9,Critical Transaction Value,1.204520e+10


In [35]:
# ============================================================
# 21. FINANCIAL RISK CONCENTRATION
# ============================================================

investigation_fraud_amount = investigation_queue.loc[
    investigation_queue["isFraud"] == 1,
    "amount"
].sum()

investigation_fraud_amount_capture = (
    investigation_fraud_amount
    / total_fraud_amount
    * 100
)

critical_fraud_amount_capture = (
    critical_fraud_amount
    / total_fraud_amount
    * 100
)

financial_risk_summary = pd.DataFrame({
    "Metric": [
        "Total Fraud Amount",
        "Investigation Queue Fraud Amount",
        "Investigation Fraud Amount Capture %",
        "Critical Fraud Amount",
        "Critical Fraud Amount Capture %"
    ],
    "Value": [
        total_fraud_amount,
        investigation_fraud_amount,
        investigation_fraud_amount_capture,
        critical_fraud_amount,
        critical_fraud_amount_capture
    ]
})

print("Financial Risk Concentration:")
display(financial_risk_summary)

Financial Risk Concentration:


,Metric,Value
0,Total Fraud Amount,1.205642e+10
1,Investigation Queue Fraud Amount,1.205573e+10
2,Investigation Fraud Amount Capture %,9.999431e+01
3,Critical Fraud Amount,1.204520e+10
4,Critical Fraud Amount Capture %,9.990698e+01


In [36]:
# ============================================================
# 22. CREATE PRIMARY RISK REASON
# ============================================================

def get_primary_risk_reason(row):

    if (
        row["signal_full_balance_transfer"]
        and row["signal_balance_depleted"]
    ):
        return "Full balance transfer + balance depleted"

    elif (
        row["signal_full_balance_transfer"]
        and row["signal_high_balance_utilization"]
    ):
        return "Full balance transfer + high utilization"

    elif row["signal_zero_amount"]:
        return "Zero-amount transaction anomaly"

    elif row["signal_large_origin_balance_change"]:
        return "Large origin balance change"

    elif row["signal_balance_depleted"]:
        return "Origin balance depleted"

    elif row["signal_high_balance_utilization"]:
        return "High origin balance utilization"

    else:
        return "Model-driven risk"


risk_intelligence["primary_risk_reason"] = (
    risk_intelligence.apply(
        get_primary_risk_reason,
        axis=1
    )
)

print("Primary Risk Reason Distribution:")
print(
    risk_intelligence["primary_risk_reason"]
    .value_counts()
)

print("\nSample Risk Intelligence:")
display(
    risk_intelligence[
        [
            "type",
            "amount",
            "fraud_probability",
            "operational_risk_tier",
            "risk_signal_count",
            "primary_risk_reason",
            "isFraud"
        ]
    ].head(20)
)

Primary Risk Reason Distribution:
primary_risk_reason
Model-driven risk                           2890954
Large origin balance change                 1940305
Origin balance depleted                     1512573
High origin balance utilization               10754
Full balance transfer + balance depleted       8008
Zero-amount transaction anomaly                  16
Full balance transfer + high utilization         10
Name: count, dtype: int64

Sample Risk Intelligence:


,type,amount,fraud_probability,operational_risk_tier,risk_signal_count,primary_risk_reason,isFraud
0,PAYMENT,9839.64,0.009906,LOW,1,Large origin balance change,0
1,PAYMENT,1864.28,0.009906,LOW,0,Model-driven risk,0
2,TRANSFER,181.00,0.991923,CRITICAL,4,Full balance transfer + balance depleted,1
3,CASH_OUT,181.00,0.991191,CRITICAL,4,Full balance transfer + balance depleted,1
4,PAYMENT,11668.14,0.009906,LOW,1,Large origin balance change,0
5,PAYMENT,7817.71,0.009906,LOW,0,Model-driven risk,0
6,PAYMENT,7107.77,0.009906,LOW,0,Model-driven risk,0
7,PAYMENT,7861.64,0.009906,LOW,1,Large origin balance change,0
8,PAYMENT,4024.36,0.013685,MEDIUM,2,Origin balance depleted,0
9,DEBIT,5337.77,0.009906,LOW,0,Model-driven risk,0


In [37]:
# ============================================================
# 23. CREATE POWER BI-READY SUMMARY TABLES
# ============================================================

# ------------------------------------------------------------
# 23.1 RISK TIER SUMMARY
# ------------------------------------------------------------

risk_tier_summary = (
    risk_intelligence
    .groupby("operational_risk_tier", observed=True)
    .agg(
        transactions=("isFraud", "size"),
        fraud_transactions=("isFraud", "sum"),
        transaction_amount=("amount", "sum"),
        fraud_amount=("amount", lambda x: x[
            risk_intelligence.loc[x.index, "isFraud"] == 1
        ].sum()),
        average_transaction_amount=("amount", "mean"),
        average_risk_score=("risk_score", "mean")
    )
    .reset_index()
)

risk_tier_summary["fraud_rate"] = (
    risk_tier_summary["fraud_transactions"]
    / risk_tier_summary["transactions"] * 100
)

risk_tier_summary["fraud_capture"] = (
    risk_tier_summary["fraud_transactions"]
    / total_fraud_transactions * 100
)


# ------------------------------------------------------------
# 23.2 TRANSACTION TYPE SUMMARY
# ------------------------------------------------------------

transaction_type_summary = (
    risk_intelligence
    .groupby("type")
    .agg(
        transactions=("isFraud", "size"),
        fraud_transactions=("isFraud", "sum"),
        transaction_amount=("amount", "sum"),
        fraud_amount=("amount", lambda x: x[
            risk_intelligence.loc[x.index, "isFraud"] == 1
        ].sum()),
        average_amount=("amount", "mean"),
        average_risk_score=("risk_score", "mean")
    )
    .reset_index()
)

transaction_type_summary["fraud_rate"] = (
    transaction_type_summary["fraud_transactions"]
    / transaction_type_summary["transactions"] * 100
)

transaction_type_summary["fraud_capture"] = (
    transaction_type_summary["fraud_transactions"]
    / total_fraud_transactions * 100
)


# ------------------------------------------------------------
# 23.3 RISK REASON SUMMARY
# ------------------------------------------------------------

risk_reason_summary = (
    risk_intelligence
    .groupby("primary_risk_reason")
    .agg(
        transactions=("isFraud", "size"),
        fraud_transactions=("isFraud", "sum"),
        transaction_amount=("amount", "sum"),
        fraud_amount=("amount", lambda x: x[
            risk_intelligence.loc[x.index, "isFraud"] == 1
        ].sum()),
        average_probability=("fraud_probability", "mean"),
        average_risk_score=("risk_score", "mean")
    )
    .reset_index()
)

risk_reason_summary["fraud_rate"] = (
    risk_reason_summary["fraud_transactions"]
    / risk_reason_summary["transactions"] * 100
)

risk_reason_summary["fraud_capture"] = (
    risk_reason_summary["fraud_transactions"]
    / total_fraud_transactions * 100
)


# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("========== RISK TIER SUMMARY ==========")
display(risk_tier_summary)

print("\n========== TRANSACTION TYPE SUMMARY ==========")
display(transaction_type_summary)

print("\n========== RISK REASON SUMMARY ==========")
display(risk_reason_summary)

========== RISK TIER SUMMARY ==========


,operational_risk_tier,transactions,fraud_transactions,transaction_amount,fraud_amount,average_transaction_amount,average_risk_score,fraud_rate,fraud_capture
0,LOW,4996706,1,7.931071e+11,1.231949e+05,1.587260e+05,0.837999,0.000020,0.012176
1,MEDIUM,1354043,4,3.378978e+11,5.628809e+05,2.495473e+05,1.407507,0.000295,0.048703
2,HIGH,3689,26,1.342893e+09,1.052842e+07,3.640264e+05,21.401016,0.704798,0.316571
3,CRITICAL,8182,8182,1.204520e+10,1.204520e+10,1.472159e+06,99.188484,100.000000,99.622550



========== TRANSACTION TYPE SUMMARY ==========


,type,transactions,fraud_transactions,transaction_amount,fraud_amount,average_amount,average_risk_score,fraud_rate,fraud_capture
0,CASH_IN,1399284,0,2.363674e+11,0.000000e+00,168920.242004,0.848559,0.000000,0.00000
1,CASH_OUT,2237500,4116,3.944130e+11,5.989202e+09,176273.964346,1.305998,0.183955,50.11567
2,DEBIT,41432,0,2.271992e+08,0.000000e+00,5483.665314,0.887454,0.000000,0.00000
3,PAYMENT,2151495,0,2.809337e+10,0.000000e+00,13057.604660,0.854047,0.000000,0.00000
4,TRANSFER,532909,4097,4.852920e+11,6.067213e+09,910647.009645,1.876067,0.768799,49.88433



========== RISK REASON SUMMARY ==========


,primary_risk_reason,transactions,fraud_transactions,transaction_amount,fraud_amount,average_probability,average_risk_score,fraud_rate,fraud_capture
0,Full balance transfer + balance depleted,8008,8008,1.054740e+10,1.054740e+10,0.991903,99.190323,100.000000,97.503957
1,Full balance transfer + high utilization,10,10,3.048522e+07,3.048522e+07,0.986583,98.658295,100.000000,0.121758
2,High origin balance utilization,10754,1,1.858557e+09,1.000000e+07,0.008642,0.864223,0.009299,0.012176
3,Large origin balance change,1940305,130,2.069445e+11,1.281037e+09,0.008524,0.852425,0.006700,1.582856
4,Model-driven risk,2890954,44,5.132035e+11,1.868354e+08,0.010507,1.050655,0.001522,0.535736
5,Origin balance depleted,1512573,4,4.118085e+11,6.598606e+05,0.009825,0.982451,0.000264,0.048703
6,Zero-amount transaction anomaly,16,16,0.000000e+00,0.000000e+00,0.989139,98.913887,100.000000,0.194813


In [38]:
# ============================================================
# 24. EXPORT RISK INTELLIGENCE DATASETS
# ============================================================

import os

# Create processed output directory if it does not exist
OUTPUT_DIR = "../data/processed"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ------------------------------------------------------------
# 24.1 FULL RISK INTELLIGENCE DATASET
# ------------------------------------------------------------

risk_intelligence_path = (
    f"{OUTPUT_DIR}/risk_intelligence.csv"
)

risk_intelligence.to_csv(
    risk_intelligence_path,
    index=False
)


# ------------------------------------------------------------
# 24.2 INVESTIGATION QUEUE
# ------------------------------------------------------------

investigation_queue_path = (
    f"{OUTPUT_DIR}/investigation_queue.csv"
)

investigation_queue.to_csv(
    investigation_queue_path,
    index=False
)


# ------------------------------------------------------------
# 24.3 POWER BI SUMMARY TABLES
# ------------------------------------------------------------

risk_tier_path = (
    f"{OUTPUT_DIR}/risk_tier_summary.csv"
)

transaction_type_path = (
    f"{OUTPUT_DIR}/transaction_type_summary.csv"
)

risk_reason_path = (
    f"{OUTPUT_DIR}/risk_reason_summary.csv"
)

risk_tier_summary.to_csv(
    risk_tier_path,
    index=False
)

transaction_type_summary.to_csv(
    transaction_type_path,
    index=False
)

risk_reason_summary.to_csv(
    risk_reason_path,
    index=False
)


# ------------------------------------------------------------
# 24.4 VERIFY FILES
# ------------------------------------------------------------

print("Export completed successfully.\n")

print("Files created:")

for file_path in [
    risk_intelligence_path,
    investigation_queue_path,
    risk_tier_path,
    transaction_type_path,
    risk_reason_path
]:
    file_size_mb = os.path.getsize(file_path) / (1024 ** 2)

    print(
        f"✓ {file_path} "
        f"({file_size_mb:.2f} MB)"
    )

Export completed successfully.

Files created:
✓ ../data/processed/risk_intelligence.csv (851.85 MB)
✓ ../data/processed/investigation_queue.csv (1.43 MB)
✓ ../data/processed/risk_tier_summary.csv (0.00 MB)
✓ ../data/processed/transaction_type_summary.csv (0.00 MB)
✓ ../data/processed/risk_reason_summary.csv (0.00 MB)


In [39]:
# ============================================================
# 25. FINAL RISK INTELLIGENCE VALIDATION
# ============================================================

print("=" * 65)
print("FINAL RISK INTELLIGENCE VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. DATASET VALIDATION
# ------------------------------------------------------------

print("\n1. DATASET")
print("-" * 65)
print(f"Total transactions: {len(risk_intelligence):,}")
print(f"Total transaction value: ₹{risk_intelligence['amount'].sum():,.2f}")
print(f"Total fraud transactions: {total_fraud_transactions:,}")
print(f"Total fraud amount: ₹{total_fraud_amount:,.2f}")


# ------------------------------------------------------------
# 2. RISK TIER VALIDATION
# ------------------------------------------------------------

print("\n2. OPERATIONAL RISK TIERS")
print("-" * 65)

print(
    risk_intelligence["operational_risk_tier"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 3. INVESTIGATION QUEUE
# ------------------------------------------------------------

print("\n3. INVESTIGATION QUEUE")
print("-" * 65)

print(f"Queue transactions: {len(investigation_queue):,}")

queue_percentage = (
    len(investigation_queue)
    / len(risk_intelligence)
    * 100
)

queue_fraud_capture = (
    investigation_queue["isFraud"].sum()
    / total_fraud_transactions
    * 100
)

print(f"Queue percentage: {queue_percentage:.4f}%")
print(
    f"Fraud transactions captured: "
    f"{investigation_queue['isFraud'].sum():,}"
)
print(f"Fraud transaction capture: {queue_fraud_capture:.4f}%")

queue_fraud_amount = investigation_queue.loc[
    investigation_queue["isFraud"] == 1,
    "amount"
].sum()

queue_fraud_amount_capture = (
    queue_fraud_amount
    / total_fraud_amount
    * 100
)

print(
    f"Fraud amount captured: "
    f"₹{queue_fraud_amount:,.2f}"
)
print(
    f"Fraud amount capture: "
    f"{queue_fraud_amount_capture:.4f}%"
)


# ------------------------------------------------------------
# 4. CRITICAL TIER
# ------------------------------------------------------------

print("\n4. CRITICAL RISK TIER")
print("-" * 65)

critical_count = (
    risk_intelligence["operational_risk_tier"] == "CRITICAL"
).sum()

critical_fraud = risk_intelligence.loc[
    risk_intelligence["operational_risk_tier"] == "CRITICAL",
    "isFraud"
].sum()

critical_fraud_rate = (
    critical_fraud
    / critical_count
    * 100
)

print(f"Critical transactions: {critical_count:,}")
print(f"Critical fraud transactions: {critical_fraud:,}")
print(f"Critical fraud rate: {critical_fraud_rate:.4f}%")
print(f"Critical fraud amount: ₹{critical_fraud_amount:,.2f}")

critical_amount_capture = (
    critical_fraud_amount
    / total_fraud_amount
    * 100
)

print(
    f"Critical fraud amount capture: "
    f"{critical_amount_capture:.4f}%"
)


# ------------------------------------------------------------
# 5. DATA QUALITY CHECKS
# ------------------------------------------------------------

print("\n5. DATA QUALITY CHECKS")
print("-" * 65)

print(
    "Missing fraud probabilities:",
    risk_intelligence["fraud_probability"].isna().sum()
)

print(
    "Missing risk tiers:",
    risk_intelligence["operational_risk_tier"].isna().sum()
)

print(
    "Missing risk reasons:",
    risk_intelligence["primary_risk_reason"].isna().sum()
)

print(
    "Duplicate transaction rows:",
    risk_intelligence.duplicated().sum()
)


# ------------------------------------------------------------
# 6. FINAL BUSINESS SUMMARY
# ------------------------------------------------------------

print("\n6. FINAL BUSINESS SUMMARY")
print("-" * 65)

print(
    f"Only {queue_percentage:.2f}% of transactions "
    f"enter the investigation queue."
)

print(
    f"The queue captures {queue_fraud_capture:.2f}% "
    f"of observed fraud transactions."
)

print(
    f"The queue captures {queue_fraud_amount_capture:.2f}% "
    f"of observed fraudulent transaction value."
)

print(
    f"The CRITICAL tier contains {critical_count:,} transactions "
    f"and captures {critical_amount_capture:.2f}% "
    f"of fraudulent value."
)

print("\n" + "=" * 65)
print("RISK INTELLIGENCE VALIDATION COMPLETE")
print("=" * 65)

FINAL RISK INTELLIGENCE VALIDATION

1. DATASET
-----------------------------------------------------------------
Total transactions: 6,362,620
Total transaction value: ₹1,144,392,944,759.77
Total fraud transactions: 8,213
Total fraud amount: ₹12,056,415,427.84

2. OPERATIONAL RISK TIERS
-----------------------------------------------------------------
operational_risk_tier
LOW         4996706
MEDIUM      1354043
HIGH           3689
CRITICAL       8182
Name: count, dtype: int64

3. INVESTIGATION QUEUE
-----------------------------------------------------------------
Queue transactions: 11,871
Queue percentage: 0.1866%
Fraud transactions captured: 8,208
Fraud transaction capture: 99.9391%
Fraud amount captured: ₹12,055,729,351.98
Fraud amount capture: 99.9943%

4. CRITICAL RISK TIER
-----------------------------------------------------------------
Critical transactions: 8,182
Critical fraud transactions: 8,182
Critical fraud rate: 100.0000%
Critical fraud amount: ₹12,045,200,936.76
Criti